# BLAST

![BLAST](https://proto-bio.github.io/proto-assets/images/tool/blast/hero.png)

This notebook demonstrates the two BLAST tools: `run_create_blast_db` builds a local BLAST database from a FASTA file, and `run_blast_search` searches sequences against either a local database or NCBI's online servers. BLAST+ binaries are downloaded automatically on first use, so no manual installation is required.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("blast")
display_overview("blast")
display_docs_section("blast", "Background")

# BLAST

[BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi) (Basic Local Alignment Search Tool) is a sequence-similarity search method maintained by the [National Center for Biotechnology Information (NCBI)](https://www.ncbi.nlm.nih.gov/). It finds regions of local similarity between a query nucleotide or protein sequence and entries in a reference database, returning ranked alignments with statistical significance scores. This toolkit exposes both the public NCBI BLAST web service and the local NCBI BLAST+ command-line distribution under a single Python interface.

BLAST ([Altschul et al., 1990](https://doi.org/10.1016/S0022-2836(05)80360-2)) performs sequence-similarity search through a heuristic algorithm that approximates the exhaustive [Smith-Waterman](https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm) local alignment at a fraction of its computational cost. The query is first broken into short fixed-length words, exact word matches are located in the database, and each match is extended in both directions until the running alignment score drops below a threshold. The statistical significance of each surviving alignment is expressed as an E-value derived from the Karlin-Altschul statistics, which represents the number of alignments with at least the observed score that would be expected to occur by chance for a database of the given size.

BLAST supports five program variants that pair query and database types appropriately. `blastn` aligns a nucleotide query against a nucleotide database. `blastp` aligns a protein query against a protein database. `blastx` translates a nucleotide query and aligns the translations against a protein database. `tblastn` aligns a protein query against a database of translated nucleotide sequences. `tblastx` translates both query and database. The toolkit's local execution mode uses the [NCBI BLAST+](https://www.ncbi.nlm.nih.gov/books/NBK279690/) command-line distribution ([Camacho et al., 2009](https://doi.org/10.1186/1471-2105-10-421)), which provides the `blastn`, `blastp`, `blastx`, `tblastn`, `tblastx`, and `makeblastdb` command-line programs that this toolkit invokes. The remote execution mode dispatches to the public [NCBI BLAST web service](https://blast.ncbi.nlm.nih.gov/Blast.cgi) through the QBLAST API.

## Available tools

In [2]:
display_available_tools("blast")

- **`run_create_blast_db()`** — Create a local BLAST database from a FASTA file
- **`run_blast_search()`** — Search sequences against BLAST databases (online or local)

### `run_create_blast_db`

`run_create_blast_db` builds a local BLAST database from a FASTA file using the `makeblastdb` utility from BLAST+. It produces a set of indexed database files whose base path can then be passed directly to `BlastSearchConfig.local_db`. Both nucleotide and protein databases are supported via the `dbtype` parameter.

In [3]:
import tempfile
from pathlib import Path

from proto_tools import (
    CreateBlastDbInput,
    CreateBlastDbConfig,
    run_create_blast_db,
)

In [4]:
# Display input docs
display_api_reference("blast", "input", "run_create_blast_db")

# Write a small FASTA with three well-characterized human globin proteins.
# The handle keeps the inputs and generated database alive across cells; the
# final cell cleans it up so nothing is left on disk.
tmp_dir_handle = tempfile.TemporaryDirectory(prefix="blast_example_")
tmp_dir = Path(tmp_dir_handle.name)

fasta_path = Path(tmp_dir) / "test_sequences.fasta"
fasta_path.write_text(
    ">sp|P69905|HBA_HUMAN Hemoglobin subunit alpha\n"
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH\n"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL\n"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR\n"
    ">sp|P68871|HBB_HUMAN Hemoglobin subunit beta\n"
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST\n"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP\n"
    "ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH\n"
    ">sp|P02144|MYG_HUMAN Myoglobin\n"
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL\n"
    "KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP\n"
    "VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG\n"
)
print(f"Wrote test FASTA to: {fasta_path}")

db_input = CreateBlastDbInput(fasta=str(fasta_path))

**Input** — `CreateBlastDbInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>fasta</code> | <code>str</code> | required | Path to the input FASTA file containing sequences to create a BLAST database from |

Wrote test FASTA to: /tmp/blast_example_itzand_x/test_sequences.fasta


In [5]:
# Display config docs
display_api_reference("blast", "config", "run_create_blast_db")

# Create a protein database with a descriptive title | see docs above for all fields
db_config = CreateBlastDbConfig(dbtype="prot", title="Test Protein DB")

**Config** — `CreateBlastDbConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>dbtype</code> | <code>Literal['nucl', 'prot']</code> | <code>'nucl'</code> | Database type: `nucl` for DNA/RNA, `prot` for amino acid. Must match the input FASTA. |
| <code>out_prefix</code> | <code>str &#124; None</code> | <code>None</code> | File-path prefix for the generated DB files; falls back to the input FASTA stem when None. |
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Descriptive DB title shown in search reports; `makeblastdb` falls back to the input file name. |
| <code>parse_seqids</code> | <code>bool</code> | <code>False</code> | Parse FASTA seq IDs so `blastdbcmd` can address sequences by ID; required for v5 taxonomy. |
| <code>hash_index</code> | <code>bool</code> | <code>False</code> | Build a hash index of sequence IDs for faster ID lookups; usually paired with `parse_seqids`. |
| <code>blastdb_version</code> | <code>Literal[4, 5]</code> | <code>5</code> | BLAST DB format version: `5` (taxonomy-aware, default since BLAST+ 2.10) or `4` (legacy). |
| <code>max_file_sz</code> | <code>str</code> | <code>'1GB'</code> | Max size per DB volume with unit suffix (e.g. `1GB`, `500MB`); `4GB` is the upstream max. |
| <code>taxid</code> | <code>int &#124; None</code> | <code>None</code> | NCBI taxonomy ID assigned to every sequence; set to tag a single-organism DB. |
| <code>extra_args</code> | <code>list[str]</code> | <code>[]</code> | Verbatim `makeblastdb` flags for fields not exposed above (e.g. `-mask_data /path`). |

In [6]:
# Run database creation
db_result = run_create_blast_db(db_input, db_config)

Running run_create_blast_db [00:00]

In [7]:
# Display output docs
display_api_reference("blast", "output", "run_create_blast_db")

# The db_path is passed directly to BlastSearchConfig.local_db for searches
print(f"Database created at: {db_result.db_path}")

**Output** — `CreateBlastDbOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>db_path</code> | <code>str</code> | required | Path to the generated BLAST database |

Database created at: /tmp/blast_example_itzand_x/test_sequences


### `run_blast_search`

`run_blast_search` performs sequence similarity search in either local or online mode. Local mode runs BLAST+ against a database created by `run_create_blast_db`, while online mode submits the query to NCBI's QBLAST service and returns hits from large public databases such as SwissProt or nr. The query accepts either a raw sequence string or a path to a FASTA file, and results are returned as a list of `BlastHit` objects with the standard 12-column tabular fields.

In [8]:
from proto_tools import (
    BlastSearchInput,
    BlastSearchConfig,
    run_blast_search,
)

In [9]:
# Display input docs
display_api_reference("blast", "input", "run_blast_search")

# Write a FASTA query file with the hemoglobin beta sequence
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

query_path = Path(tmp_dir) / "query.fasta"
query_path.write_text(f">query_hbb\n{query_seq}\n")

search_input = BlastSearchInput(query=str(query_path))

**Input** — `BlastSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query</code> | <code>str</code> | required | Query sequence string or path to a FASTA file containing query sequence(s) |
| <code>query_type</code> | <code>Literal['sequence', 'fasta_path']</code> | <code>'sequence'</code> | Auto-inferred query type: 'sequence' for raw string, 'fasta_path' for a file path. |

In [10]:
# Display config docs
display_api_reference("blast", "config", "run_blast_search")

# Local blastp search against the database we just built | see docs above for all fields
search_config = BlastSearchConfig(
    search_mode="local",
    program="blastp",
    local_db=db_result.db_path,
    num_threads=2,
)

**Config** — `BlastSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>search_mode</code> | <code>Literal['online', 'local']</code> | <code>'online'</code> | `online` queries NCBI's QBLAST servers; `local` runs BLAST+ CLI against a local database. |
| <code>program</code> | <code>Literal['blastn', 'blastp', 'blastx', 'tblastn', 'tblastx']</code> | <code>'blastn'</code> | BLAST algorithm (blastn/blastp/blastx/tblastn/tblastx). |
| <code>database</code> | <code>Literal['nt', 'nr', 'refseq_rna', 'refseq_protein', 'swissprot', 'pdb', 'pataa', 'patnt']</code> | <code>'nt'</code> | NCBI database to search against (e.g. `nt`, `nr`, `swissprot`, `pdb`). |
| <code>entrez_query</code> | <code>str &#124; None</code> | <code>None</code> | Restrict the online search with an Entrez query (e.g. `Homo sapiens[Organism]`). |
| <code>hitlist_size</code> | <code>int &#124; None</code> | <code>None</code> | Number of database sequences to return; NCBI defaults to 50 when unset. |
| <code>megablast</code> | <code>bool &#124; None</code> | <code>None</code> | Use the MegaBLAST algorithm for `blastn` (faster, optimized for highly similar sequences). |
| <code>local_db</code> | <code>str &#124; None</code> | <code>None</code> | Path to a local BLAST database stem (no file extensions). Required for local mode. |
| <code>num_threads</code> | <code>int</code> | <code>4</code> | CPU threads for local BLAST search (upstream BLAST+ default is 1). |
| <code>evalue</code> | <code>float &#124; None</code> | <code>None</code> | Expectation value threshold for reporting hits; BLAST+ default is 10.0 when unset. |
| <code>word_size</code> | <code>int &#124; None</code> | <code>None</code> | Length of the initial exact match; BLAST+ defaults: 28 (megablast), 11 (blastn), 3 (blastp). |
| <code>gapopen</code> | <code>int &#124; None</code> | <code>None</code> | Cost to open a gap; BLAST+ defaults: 5 (blastn), 11 (protein programs). |
| <code>gapextend</code> | <code>int &#124; None</code> | <code>None</code> | Cost to extend a gap; BLAST+ defaults 2 (blastn), 1 (protein). Not for tblastx. |
| <code>matrix</code> | <code>Literal['BLOSUM45', 'BLOSUM50', 'BLOSUM62', 'BLOSUM80', 'BLOSUM90', 'PAM30', 'PAM70', 'PAM250'] &#124; None</code> | <code>None</code> | Substitution matrix for protein alignments; BLAST+ default is BLOSUM62. Not applicable to blastn. |
| <code>reward</code> | <code>int &#124; None</code> | <code>None</code> | Match reward (blastn only); BLAST+ defaults 1 (megablast), 2 (blastn). |
| <code>penalty</code> | <code>int &#124; None</code> | <code>None</code> | Mismatch penalty (blastn, ≤0); BLAST+ defaults -2 (megablast), -3 (others). |
| <code>threshold</code> | <code>int &#124; None</code> | <code>None</code> | Minimum word score for the BLAST lookup table (protein only); BLAST+ default is 11 for blastp. |
| <code>comp_based_stats</code> | <code>Literal[0, 1, 2, 3] &#124; None</code> | <code>None</code> | Composition-based scoring (protein): 0=off, 1=stats, 2=adjust (default), 3=unconditional. |
| <code>max_target_seqs</code> | <code>int &#124; None</code> | <code>None</code> | Maximum number of aligned sequences to keep; BLAST+ default is 500. |
| <code>perc_identity</code> | <code>float &#124; None</code> | <code>None</code> | Minimum percent identity for reported alignments (0-100); most useful with blastn. |
| <code>qcov_hsp_perc</code> | <code>float &#124; None</code> | <code>None</code> | Minimum query coverage percentage per HSP (0-100). |
| <code>soft_masking</code> | <code>bool &#124; None</code> | <code>None</code> | Apply filter as soft masks (seeding only); BLAST+ defaults True (blastn), False (protein). |
| <code>lcase_masking</code> | <code>bool &#124; None</code> | <code>None</code> | Treat lowercase letters in the FASTA input as masked. |
| <code>dust</code> | <code>str &#124; None</code> | <code>None</code> | Nucleotide low-complexity filter: `yes`, `no`, or `level window linker` (default `20 64 1`). |
| <code>seg</code> | <code>str &#124; None</code> | <code>None</code> | Protein low-complexity filter: `yes`, `no`, or `window locut hicut` (blastp default `no`). |
| <code>task</code> | <code>Literal['megablast', 'dc-megablast', 'blastn', 'blastn-short', 'blastp', 'blastp-short', 'blastp-fast', 'blastx', 'blastx-fast', 'tblastn', 'tblastn-fast'] &#124; None</code> | <code>None</code> | Task preset (e.g. `megablast`, `blastp-fast`, `blastx-fast`); flips multiple defaults at once. |
| <code>ungapped</code> | <code>bool &#124; None</code> | <code>None</code> | Perform ungapped alignment only. |
| <code>strand</code> | <code>Literal['both', 'plus', 'minus'] &#124; None</code> | <code>None</code> | Query strand(s) to search; BLAST+ default is `both`. Only for blastn, blastx, tblastx. |
| <code>query_gencode</code> | <code>int &#124; None</code> | <code>None</code> | Genetic code for translating the query (blastx/tblastx only); BLAST+ default 1 (Standard). |
| <code>db_gencode</code> | <code>int &#124; None</code> | <code>None</code> | Genetic code for translating database sequences (tblastn/tblastx only); BLAST+ default 1. |
| <code>extra_args</code> | <code>list[str]</code> | <code>[]</code> | Verbatim BLAST+ CLI tokens for niche flags (e.g. `['-max_hsps', '1']`). Local mode only. |

In [11]:
# Run the local BLAST search
result = run_blast_search(search_input, search_config)

Running run_blast_search [00:00]

In [12]:
# Display output docs
display_api_reference("blast", "output", "run_blast_search")

# HBB should self-hit at 100% identity; HBA should appear at ~43% identity
print(f"Found {result.num_hits} hits\n")
for hit in result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

**Output** — `BlastSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>hits</code> | <code>list[BlastHit]</code> | <code>[]</code> | BLAST alignment hits |

**`BlastHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>qseqid</code> | <code>str</code> | required | Query sequence ID |
| <code>sseqid</code> | <code>str</code> | required | Subject sequence ID |
| <code>pident</code> | <code>float</code> | required | Percentage of identical matches |
| <code>length</code> | <code>int</code> | required | Alignment length |
| <code>mismatch</code> | <code>int</code> | required | Number of mismatches |
| <code>gapopen</code> | <code>int</code> | required | Number of gap openings |
| <code>qstart</code> | <code>int</code> | required | 1-indexed inclusive start in query |
| <code>qend</code> | <code>int</code> | required | 1-indexed inclusive end in query |
| <code>sstart</code> | <code>int</code> | required | 1-indexed inclusive subject start (sstart > send means minus strand for nucleotide subjects) |
| <code>send</code> | <code>int</code> | required | 1-indexed inclusive subject end |
| <code>evalue</code> | <code>float</code> | required | Expect value; smaller values are more significant |
| <code>bitscore</code> | <code>float</code> | required | Alignment bit score; higher is better |

Found 2 hits

  sp|P68871|HBB_HUMAN: 100.0% identity, e-value=1.4e-111, bitscore=301.0
  sp|P69905|HBA_HUMAN: 43.4% identity, e-value=5.4e-38, bitscore=114.0


#### Online search

For searches against large public databases, set `search_mode="online"`. The query is submitted to NCBI's QBLAST service and requires an internet connection. The raw sequence string can be passed directly to `BlastSearchInput` without writing a FASTA file.

In [13]:
# Online search accepts a raw sequence string directly
online_input = BlastSearchInput(query=query_seq)
online_config = BlastSearchConfig(
    search_mode="online",
    program="blastp",
    database="swissprot",
    hitlist_size=5,
)

online_result = run_blast_search(online_input, online_config)
print(f"Found {online_result.num_hits} hits\n")
for hit in online_result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

Found 5 hits

  P68871: 100.0% identity, e-value=5.8e-106, bitscore=301.2
  P02024: 99.3% identity, e-value=2.0e-105, bitscore=300.1
  P02025: 98.6% identity, e-value=3.0e-103, bitscore=294.3
  P02032: 97.3% identity, e-value=2.4e-102, bitscore=292.0
  P19885: 95.9% identity, e-value=4.3e-102, bitscore=291.6


## Export Results

Search results can be exported to CSV or JSON for downstream analysis. CSV produces one row per hit with the standard 12 tabular fields; JSON preserves the same structure for programmatic consumption.

In [14]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="blast_search_local_results", export_path=output_dir, file_format="csv")
online_result.export(name="blast_search_online_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output


In [15]:
# Clean up the temporary directory holding the example inputs and database
tmp_dir_handle.cleanup()